In [1]:
import numpy as np
from scipy import ndimage
import matplotlib.pyplot as plt
from caiman.source_extraction.cnmf.deconvolution import constrained_foopsi
import tiffile
import seaborn as sns
from scipy import stats
import os
import networkx as nx
import h5py

/beegfs_hdd/data/nfs_share/users/guiyun/nishome/brainnetwork/src/CaImAn/caiman/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-03-05 15:14:23.052168: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:

base_dir =  "../../Micedata/"
data_paths = ["M21_1107", "M71_1024", "M73_1128", "M77_1028", "M77_1031", "M77_1107", "M78_1017", "M79_1128", "M91_1017"]

In [5]:
for dir in data_paths:
    # print("开始处理数据...")
    data_path = base_dir + dir
    mat_file = os.path.join(data_path, 'wholebrain_output.mat')
    if not os.path.exists(mat_file):
        raise ValueError(f"未找到神经数据文件: {mat_file}")
    try:
        data = h5py.File(mat_file, 'r')
    except Exception as e:
        raise ValueError(f"无法读取mat文件: {mat_file}，错误信息: {e}")

    # 检查关键数据集是否存在
    if 'whole_trace_ori' not in data or 'whole_center' not in data:
        raise ValueError("mat文件缺少必要的数据集（'whole_trace_ori' 或 'whole_center'）")

    # ==========神经数据================
    neuron_data = data['whole_trace_ori']
    # 转化成numpy数组
    neuron_data = np.array(neuron_data)
    # print(f"原始神经数据形状: {neuron_data.shape}")

    # 只做基本的数据清理：移除NaN和Inf
    neuron_data = np.nan_to_num(neuron_data, nan=0.0, posinf=0.0, neginf=0.0)
    neuron_pos = data['whole_center']

    print("神经数据形状:", neuron_data.shape)
    # %%
    # =========== 第一步 提取仅有正值的神经元==================
    # 带负值的神经元索引
    mask = np.any(neuron_data <= 0, axis=0)   # 每列是否存在 <=0
    keep_idx = np.where(~mask)[0]

    # 从数据中删除这些列
    neuron_data = neuron_data[:, keep_idx]
    neuron_pos = neuron_pos[:, keep_idx]

    print("神经数据形状:", neuron_data.shape)
    # %%
    # =========== 第二步 计算 dF/F0 ==================
    win_size = 151
    T, N = neuron_data.shape
    F0_dynamic = np.zeros((T, N), dtype=float)
    for i in range(N):
        # ndimage.percentile_filter 输出每帧的窗口百分位值
        F0_dynamic[:, i] = ndimage.percentile_filter(neuron_data[:, i], percentile=8, size=win_size, mode='reflect')
    deltaF_over_F0 = (neuron_data - F0_dynamic) / F0_dynamic

    # %% =========== 第三步 对 dF/F 进行反卷积 ==================
    deconvolved_data = np.zeros_like(deltaF_over_F0)

    for i in range(N):
        if i % 100 == 0:
            print(f"处理进度: {i}/{N}")
        try:
            # 对每个神经元进行反卷积
            # p=1 对应4Hz慢信号，g会自动估计
            c, bl, c1, g, sn, sp, lam = constrained_foopsi(
                fluor=deltaF_over_F0[:, i],
                p=1,  # AR(1)模型适合慢信号
                method_deconvolution='oasis',  # 最快的方法
                optimize_g=5  # 自动优化时间常数
            )
            # 使用去卷积后的spikes
            deconvolved_data[:, i] = sp
        except Exception as e:
            print(f"神经元{i}反卷积失败: {e}")
            deconvolved_data[:, i] = deltaF_over_F0[:, i]  # 失败时保留原始dF/F
    # %% 保存文件
    from scipy.io import savemat
    # %% =========== 第四步 保存处理后的数据到MAT文件 ==================
    from scipy.io import savemat

    # 整理需要保存的数据（构建字典，key为MATLAB中变量名）
    save_dict = {
        # 原始数据（过滤后）
        'whole_center': neuron_pos,    # 对应过滤后的神经元位置
        # 处理后的数据
        'deltaF_over_F0': deltaF_over_F0,     # dF/F0信号
        'whole_trace_ori': deconvolved_data,  # 反卷积后的峰值信号
    }

    # 定义保存路径（与原始mat文件同目录，添加"_processed"后缀区分）
    save_filename = os.path.join(data_path, f'wholebrain_processed.mat')

    try:
        # 保存MAT文件（format='5'兼容MATLAB R2006b及以上版本）
        savemat(
            file_name=save_filename,
            mdict=save_dict,
            format='5',  # 推荐格式，兼容性最好
            do_compression=True  # 启用压缩，减少文件体积
        )

    except Exception as e:
        raise ValueError(f"❌ 保存MAT文件失败：{e}")

神经数据形状: (8402, 23374)
神经数据形状: (8402, 23353)
处理进度: 0/23353
处理进度: 100/23353
处理进度: 200/23353
处理进度: 300/23353
处理进度: 400/23353
处理进度: 500/23353
处理进度: 600/23353
处理进度: 700/23353
处理进度: 800/23353
处理进度: 900/23353
处理进度: 1000/23353
处理进度: 1100/23353
处理进度: 1200/23353
处理进度: 1300/23353
处理进度: 1400/23353
处理进度: 1500/23353
处理进度: 1600/23353
处理进度: 1700/23353
处理进度: 1800/23353
处理进度: 1900/23353
处理进度: 2000/23353
处理进度: 2100/23353
处理进度: 2200/23353
处理进度: 2300/23353
处理进度: 2400/23353
处理进度: 2500/23353
处理进度: 2600/23353
处理进度: 2700/23353
处理进度: 2800/23353
处理进度: 2900/23353
处理进度: 3000/23353
处理进度: 3100/23353
处理进度: 3200/23353
处理进度: 3300/23353
处理进度: 3400/23353
处理进度: 3500/23353
处理进度: 3600/23353
处理进度: 3700/23353
处理进度: 3800/23353
处理进度: 3900/23353
处理进度: 4000/23353
处理进度: 4100/23353
处理进度: 4200/23353
处理进度: 4300/23353
处理进度: 4400/23353
处理进度: 4500/23353
处理进度: 4600/23353
处理进度: 4700/23353
处理进度: 4800/23353
处理进度: 4900/23353
处理进度: 5000/23353
处理进度: 5100/23353
处理进度: 5200/23353
处理进度: 5300/23353
处理进度: 5400/23353
处理进度: 5500/23353
处理进度: 5600/23353